# Functional Test 2: Topic Graph (the `topicgraph` native executable)

## Summary

- The **builder of the Topic Graph is the native executable `topicgraph`** (.NET), distributed from `Azure/foundry-iq-tool-releases`. foundry-iq-skills pins release v1.0.10 by SHA-256. The executable runs **on your machine**, reads the chunk vectors stored in a Foundry IQ index, and builds the topic hierarchy. The plugin's Python only writes the configuration JSON, launches the executable, and reads its output. This notebook launches that executable **directly** and tests its features, including the integration with Foundry IQ and the models.
- The input is two text files in the same folder (`Thornwick Castle - Hanapedia.txt` and `Elowen Park - Hanapedia.txt`). Inside the notebook, you split them into chunks, create embeddings, and load them into a **small, test-only source index** (`ft-topic-src`). The notebook does not depend on the other functional tests (Entity Graph).
- During the build only, the executable **adds a temporary scoring profile to the source index and deletes it when it finishes** (it does not change documents or fields). This "restoration" is tested as well.
- The output stays in `.functional-test/topicgraph-out`. You can use it as the **input of Functional Test 3 (Ontology)**.
- **Chapter 9 can register the Topic Graph in Foundry IQ, read it back, visualize the live data, and explore it with GQL.** Enable it with `RUN_FOUNDRY_IQ = True`; the first write needs a separate approval through `CONFIRM_TOPIC_WRITES`. It does not depend on the other notebooks.
- Section 6.5 **displays the parent-child relationships of all topics as a graph**, and 6.6 **displays the hierarchy and the share of member chunks as a topic wheel**. Both use already generated data and make no additional model calls or writes to Foundry IQ.
- **If an existing output is found, the notebook defaults to a read-only resume mode.** Even with `Run All`, it does not recreate the index, generate embeddings, build, or clean up. Set `USE_EXISTING_OUTPUT` in 1.2 to `False` only when you want to create a new build.

### Quick start from an existing output

1. If you restarted the kernel, run **the settings cell in 1.2, then the output loading cell in 5.0**. The first time, also run the install cell in Chapter 1.
2. In **5.1 and 6.0 to 6.6**, you can review the saved run record, memberships, topic graph, and wheel. If you only need the graph, you can run 6.5 and then 6.6 directly after 5.0.
3. In **Chapter 10**, check the results of the verification that was actually performed this time. Tests that exist only for new builds are not counted as rerun.

The required existing output is `.functional-test/topicgraph-out/topicGraph`. Loading the local output needs no `.env`, no Azure authentication, no original text, and no executable. If an old output lacks the original chunk text and embeddings, 6.2 shows chunk IDs and the chunk PCA in 6.3 is skipped. New builds also save them for resuming.

> Source: [Azure/foundry-iq-tool-releases (TopicGraph releases)](https://github.com/Azure/foundry-iq-tool-releases) / [Topic Graph build: Plan and apply (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md) / [Topic Graph build (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/topic-graph-build.md)

## Resources created and related components

**The notebook creates the source index, and `topicgraph`, running on your machine, generates the topic structure and the output files.** Unlike Entity Graph, this is not a setup in which Foundry IQ automatically generates ingestion components from a knowledge source. Blue marks what the notebook creates and runs, green marks what `topicgraph` generates, and gray marks existing inputs, services, and the distributed executable. Solid lines show the main processing and data flow; dashed lines show configuration, model calls, and temporary changes.

```mermaid
flowchart TB
    subgraph LOCAL["Your machine"]
        TXT["Two existing article texts"]
        NB["Notebook<br/>T1: clean, chunk, get embeddings<br/>T2: create the index"]
        CFG["Configuration file<br/>topicgraph-config.json<br/>T3: saved by the notebook"]
        CLI["topicgraph (.NET, pinned version)<br/>T3: preflight / T4: build<br/>T7: build --resume"]
        subgraph OUT["Local output: OUT"]
            STRUCT["Topic hierarchy, memberships, labels<br/>Structure and labels of topicGraph<br/>Structure and labels copied to ontology"]
            EXPORT["Export<br/>releases<br/>Schema and graph documents"]
            STATE["Run record and checkpoints<br/>build-summary.json<br/>run-manifest.json"]
        end
        TXT --> NB
        CFG -. loads run settings .-> CLI
        CLI --> STRUCT
        CLI --> EXPORT
        CLI <-->|save and resume progress| STATE
    end

    subgraph SEARCH["Foundry IQ (existing service)"]
        IDX[("Source index<br/>ft-topic-src<br/>text + snippet_vector")]
        TI[("Published topic index<br/>Chapter 9: register, read back, GQL")]
        PROFILE["Temporary scoring profile<br/>inside the source index definition<br/>T6: verify deletion and restoration"]
    end
    subgraph AOAI["Azure OpenAI (existing model deployments)"]
        EMB["Embedding model<br/>T1: vectorize the chunks"]
        CHAT["Chat model<br/>Stage 7: label the topics"]
    end

    EXPORT -->|Chapter 9: register after explicit approval| TI
    TI --> READ["Visualize and explore the registered hierarchy"]
    NB -->|T2: upload text and vectors| IDX
    IDX -->|chunks, vectors, nearest-neighbor results| CLI
    NB -. request embeddings .-> EMB
    CLI -. request title and description generation .-> CHAT
    CLI -. added during the build, deleted at the end .-> PROFILE

    classDef notebook fill:#dbeafe,stroke:#2563eb,color:#172554
    classDef generated fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef existing fill:#f1f5f9,stroke:#64748b,color:#0f172a
    class NB,CFG,IDX,TI,READ notebook
    class STRUCT,EXPORT,STATE,PROFILE generated
    class TXT,CLI,EMB,CHAT existing
    style PROFILE stroke-dasharray:5 5
```

- **Where it runs and who generates what:** `topicgraph` is the executable that T0 downloads (or reuses) at the pinned version and whose SHA-256 it verifies. Topics are built locally. The executable calls Foundry IQ to get nearest neighbors and calls the chat model to generate labels. The temporary scoring profile is not an independent Foundry IQ resource; it is added to the definition of `ft-topic-src`.
- **What this test does not create:** It creates no knowledge source, knowledge base, data source, indexer, or skillset. The text and vectors are uploaded directly with the SDK. `blob_url` is a field name used to identify the document; this test does not upload the articles to Blob Storage.
- **Output and downstream use:** `OUT` is `.functional-test\topicgraph-out`. The structure, labels, exports, and checkpoints are saved under `topicGraph`, and the structure and labels under `ontology` can be handed to Functional Test 3. The checkpoints are local files for resuming; they are not an enrichment cache. **For a local check only, no graph documents are uploaded.** If you explicitly enable it in Chapter 9, the graph documents are uploaded to a dedicated topic index and read back. Connecting them to a search profile is a separate step.

> Source: [Topic Graph native executable and verification](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/native-runtime.md) / [Distinguishing the Topic Graph build from the upload to an index](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md) / [Prerequisites and steps for connecting a Topic Graph to search](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

## Test flow

```mermaid
flowchart LR
    TXT["Two texts"] --> CH["Chunking"] --> EMB["Embeddings<br/>Azure OpenAI"] --> IDX[("Source index<br/>ft-topic-src")]
    IDX --> PF["preflight<br/>read-only"] --> BD["build --fresh<br/>stages 0 to 9"] --> OUT[("Output folder")]
    BD -. "Temporary scoring<br/>profile" .-> IDX
    OUT --> RD["Read the results<br/>topics / memberships / labels"]
    OUT --> RS["build --resume<br/>verify reuse"]
    OUT --> PUB["Register in Foundry IQ after approval"] --> LIVE["Read back, visualize, GQL"]
```

## Test list

| ID | Target | Pass condition |
| --- | --- | --- |
| T0 | Executable | It is found (or the pinned version can be downloaded). `--version` matches the pinned version. The SHA-256 matches the plugin's record |
| T1 | Input preparation | At least 20 chunks are created. The embedding dimensions are uniform |
| T2 | Source index | The document count matches. The vector field is stored, retrievable, and uses HNSW with cosine similarity |
| T3 | `preflight` | It succeeds. The document count and dimensions match. The temporary change is announced in advance |
| T4 | `build` | It succeeds with `--fresh` the first time, or with `--resume` for an interrupted build. All stages 0 to 9 are completed or reused. The cumulative number of kNN calls matches the chunk count. The model is called. There are no failed calls |
| T5 | Output contents | The latest release is `exported`. Every chunk belongs to a topic. There are at least 2 leaf topics. The membership total equals the chunk count. Every topic has a title |
| T6 | Restoration of the temporary change | The scoring profile is deleted, and the document count and the vector field settings are unchanged |
| T7 | `build --resume` | It reuses the completed build, does not call the model again, and the release does not change |

## Prerequisites

| Item | Details |
| --- | --- |
| Authentication | You are signed in with the Azure CLI (`az login`). The executable calls Foundry IQ and the models with `defaultAzureCredential` (no interactive browser) |
| Roles (guideline) | Foundry IQ: Search Service Contributor (create the index and temporarily add the scoring profile), Search Index Data Contributor (upload documents). Azure OpenAI: Cognitive Services OpenAI User |
| `.env` | `AZURE_SEARCH_ENDPOINT`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` (and optionally `AZURE_OPENAI_MODEL`) |
| OS | The notebook is intended to be verified on Windows x64 / arm64. On Linux and macOS, set the executable's path in the environment variable `TOPICGRAPH_EXE` (the plugin supports Windows and Linux) |

## 1. Setup

### 1.1 Install packages

This notebook uses `azure-search-documents` 12.2.0b1, which includes the knowledge graph models and APIs (it is used to create the index and upload documents).

In [ ]:
%pip install -q azure-search-documents==12.2.0b1 azure-identity python-dotenv pandas matplotlib requests networkx plotly

### 1.2 Settings

`check()` records the result of each test in `results`. Even if a test fails, the following cells keep running, and all results are displayed together at the end.

The main parameters are as follows.

| Name | Meaning |
| --- | --- |
| `USE_EXISTING_OUTPUT` | It is set to `True` automatically if the output folder exists, and the notebook resumes in read-only mode. If you explicitly set it to `False`, the create path runs, including deleting and recreating the index |
| `OUT` | The saved output, or the destination of a new build. The default is `.functional-test/topicgraph-out` under the notebook folder |
| `CHUNK_CHARS` | The approximate number of characters per chunk. The smaller it is, the more chunks you get, which makes it easier to check how topics are split |
| `MAX_LEAF_SIZE` | The target size of a leaf topic (the target for subdividing with Leiden, minimum 2). The default is 100, but the two articles yield only about 45 chunks, so it is set small to force a hierarchy |

In [ ]:
import base64
import hashlib
import json
import os
import platform
import shutil
import struct
import subprocess
import time
import zipfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

matplotlib.rcParams["font.sans-serif"] = ["Segoe UI", "Arial", "DejaVu Sans"]

FT = "ft-topic"
SRC_INDEX = f"{FT}-src"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "functional-test-2-topic-graph.ipynb").is_file() and (NOTEBOOK_DIR / "graph" / "functional-test-2-topic-graph.ipynb").is_file():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "graph"
WORK = (NOTEBOOK_DIR / ".functional-test").resolve()
OUT = WORK / "topicgraph-out"      # Functional Test 3 looks here by default
USE_EXISTING_OUTPUT = (OUT / "topicGraph").is_dir()
CHUNK_CHARS = 400
MAX_LEAF_SIZE = 8
RUN_CLEANUP = False                # If True, the index and the working folder are deleted at the end

results = []
if not USE_EXISTING_OUTPUT:
    from azure.identity import DefaultAzureCredential

    load_dotenv(NOTEBOOK_DIR / ".env", override=True)
    credential = DefaultAzureCredential()
    SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
    AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
    CHAT_MODEL = os.environ.get("AZURE_OPENAI_MODEL", CHAT_DEPLOYMENT)
    EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
    WORK.mkdir(exist_ok=True)


def check(case, ok, detail=""):
    results.append({"Test": case, "Result": "PASS" if ok else "FAIL", "Detail": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


print("Loading the existing output" if USE_EXISTING_OUTPUT else "New build / resuming an unfinished build", "|", OUT)
print("Next is the output loading cell in 5.0." if USE_EXISTING_OUTPUT else "Chapters 2 to 5 create the input and the build.")

## 2. T0: Verify the executable

`topicgraph` is a self-contained .NET executable. In `native-release.json`, the plugin pins the repository (`Azure/foundry-iq-tool-releases`), the tag (`v1.0.10`), and the SHA-256 of the asset for each platform. It checks the digest of the downloaded distribution before extracting it, and records the SHA-256 of the extracted executable in `receipt.json`.

Here, the executable is located in the following order.

1. The environment variable `TOPICGRAPH_EXE` (any path)
2. The plugin cache (`%LOCALAPPDATA%\foundry-iq-skills\runtimes\topicgraph\native\win-x64\v*\extracted\topicgraph.exe`)
3. If neither exists, the pinned `win-x64` version is downloaded from GitHub Releases, **checked against the pinned digest**, and extracted

`topicgraph --version` needs no configuration, no authentication, and no access to any service.

> Source: [Azure/foundry-iq-tool-releases](https://github.com/Azure/foundry-iq-tool-releases) / [native-release.json and native_download.py (topicgraph_skills wheel)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
if USE_EXISTING_OUTPUT:
    print("Existing output mode: the executable is not downloaded or launched.")
else:
    PIN_VERSION = "1.0.10"
    PIN_URL = f"https://github.com/Azure/foundry-iq-tool-releases/releases/download/v{PIN_VERSION}/topicGraphBuildTool-{PIN_VERSION}-win-x64.zip"
    PIN_SHA256 = "b0873e104f79f021646ba310b769dcfd21435ad2f30759f7c2be76ead0661849"   # native-release.json (win-x64) in foundry-iq-skills

    cache_root = Path(os.environ.get("FOUNDRY_IQ_RUNTIME_CACHE") or (Path(os.environ.get("LOCALAPPDATA", Path.home() / ".cache")) / "foundry-iq-skills" / "runtimes"))
    EXE = Path(os.environ["TOPICGRAPH_EXE"]) if os.environ.get("TOPICGRAPH_EXE") else None
    if EXE is None:
        cached = sorted(cache_root.glob("topicgraph/native/win-x64/v*/extracted/topicgraph.exe"))
        EXE = cached[-1] if cached else None

    if EXE is None:
        assert platform.system() == "Windows", "Set TOPICGRAPH_EXE to the path of the topicgraph executable"
        archive = WORK / "native" / f"topicGraphBuildTool-{PIN_VERSION}-win-x64.zip"
        archive.parent.mkdir(parents=True, exist_ok=True)
        archive.write_bytes(requests.get(PIN_URL, timeout=300).content)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == PIN_SHA256, "The digest of the downloaded distribution does not match the pinned value"
        zipfile.ZipFile(archive).extractall(archive.parent / "extracted")
        EXE = archive.parent / "extracted" / "topicgraph.exe"

    check("T0-1 The executable exists", EXE.is_file(), EXE)
    version = subprocess.run([str(EXE), "--version"], capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
    check("T0-2 --version matches the pinned version", version == PIN_VERSION, version)

    receipt_path = EXE.parent.parent / "receipt.json"
    actual_sha = hashlib.sha256(EXE.read_bytes()).hexdigest().upper()
    if receipt_path.is_file():
        recorded = json.loads(receipt_path.read_text(encoding="utf-8"))["binary_sha256"].upper()
        check("T0-3 The SHA-256 of the executable matches the plugin's record", actual_sha == recorded, actual_sha[:16])
    else:
        print("The plugin's record (receipt.json) is missing, so the SHA-256 comparison was skipped. SHA-256 of the executable:", actual_sha)

## 3. Prepare the input

### 3.1 Why build your own source index

`topicgraph` reads **the chunk vectors stored in a Foundry IQ index**. The requirements are as follows (from `topicgraph --help` and the plugin's `native_source_policy`).

- The vector field is a **stored and retrievable float32** field that uses HNSW with cosine similarity (or inner product).
- A key field (`uid`) exists. Specify the text field (`snippet`) for labeling. The document source (`blob_url`) is an optional field for tracking.

The content index that an Entity Graph knowledge source creates also meets these conditions. However, so that this test does not depend on Entity Graph, **you build an index of the same shape yourself from the text**. HNSW is a graph-based index for nearest-neighbor search.

> Source: [Efficient and robust approximate nearest neighbor search using HNSW graphs (Malkov and Yashunin)](https://arxiv.org/abs/1603.09320) / [Plan and apply (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md)

### 3.2 Clean the text and split it into chunks

Both files have the `.txt` extension, but their content is Markdown with headings, paragraphs, and tables. Line breaks are stored as actual line breaks, and there are no page-information HTML comments or escaped line breaks. The text goes through the existing cleaning step, and paragraphs are grouped with `CHUNK_CHARS` as the guide to create chunks.

In [ ]:
import re

DATA_DIR = NOTEBOOK_DIR
SAMPLES = {"Thornwick Castle - Hanapedia.txt": "thornwick-castle", "Elowen Park - Hanapedia.txt": "elowen-park"}


def clean_text(raw):
    text = raw.replace("\\n", "\n").replace('\\"', '"')
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"<tr>(.*?)</tr>", lambda m: " / ".join(re.findall(r"<td>(.*?)</td>", m.group(1), flags=re.S)) + "\n", text, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def chunk_text(text, target):
    chunks, buf = [], ""
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if para and buf and len(buf) + len(para) > target:
            chunks.append(buf.strip())
            buf = ""
        buf += para + "\n\n" if para else ""
    return chunks + ([buf.strip()] if buf.strip() else [])


if USE_EXISTING_OUTPUT:
    print("Existing output mode: the source text is not loaded or rechunked.")
else:
    chunks = []
    for src, slug in SAMPLES.items():
        for n, body in enumerate(chunk_text(clean_text((DATA_DIR / src).read_text(encoding="utf-8")), CHUNK_CHARS)):
            chunks.append({"uid": f"{FT}-{slug}-{n:03d}", "blob_url": f"{slug}.txt", "snippet": body})

    chunk_df = pd.DataFrame(chunks)
    check("T1-1 At least 20 chunks are created", len(chunk_df) >= 20, f"{len(chunk_df)} chunks")
    check("T1-2 There are no empty chunks", bool((chunk_df["snippet"].str.len() > 0).all()), f"shortest: {chunk_df['snippet'].str.len().min()} characters")
    display(chunk_df.groupby("blob_url")["snippet"].agg(Chunks="count", Avg_chars=lambda s: round(s.str.len().mean())))
    display(chunk_df.head(3))

### 3.3 Create the embeddings

An Azure OpenAI embedding deployment turns the chunks into vectors (16 at a time). The model determines the vector dimension, so you read it from the response. The same vectors are also used for the visualization that follows (principal component analysis).

In [ ]:
def embed(texts):
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    response = requests.post(f"{AOAI_ENDPOINT}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version=2024-10-21",
                             headers={"Authorization": f"Bearer {token}"}, json={"input": texts}, timeout=120)
    response.raise_for_status()
    return [d["embedding"] for d in sorted(response.json()["data"], key=lambda d: d["index"])]


if USE_EXISTING_OUTPUT:
    print("Existing output mode: embeddings are not generated.")
else:
    vectors = [v for i in range(0, len(chunks), 16) for v in embed([c["snippet"] for c in chunks[i:i + 16]])]
    DIMS = len(vectors[0])
    check("T1-3 All embeddings have the same dimension", all(len(v) == DIMS for v in vectors) and len(vectors) == len(chunks), f"{len(vectors)} vectors x {DIMS} dimensions")

### 3.4 T2: Create the source index

Make `uid` the key (filterable), `snippet` the full-text searchable text, and `snippet_vector` a vector field that is **stored and retrievable** (HNSW, cosine). If an index with the same name remains from a previous run, it is deleted and recreated (this name is dedicated to this test). After creating it, you read it back to confirm that the settings were saved as intended.

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    HnswParameters,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    VectorSearch,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

if USE_EXISTING_OUTPUT:
    print("Existing output mode: the index is not deleted, created, or uploaded to.")
else:
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential)
    if SRC_INDEX in list(index_client.list_index_names()):
        index_client.delete_index(SRC_INDEX)

    index_client.create_index(SearchIndex(
        name=SRC_INDEX,
        fields=[
            SearchField(name="uid", type=SearchFieldDataType.String, key=True, filterable=True),
            SearchField(name="blob_url", type=SearchFieldDataType.String, filterable=True),
            SearchField(name="snippet", type=SearchFieldDataType.String, searchable=True),
            SearchField(name="snippet_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single), searchable=True,
                        vector_search_dimensions=DIMS, vector_search_profile_name="vp", hidden=False, stored=True),
        ],
        vector_search=VectorSearch(
            algorithms=[HnswAlgorithmConfiguration(name="hnsw", parameters=HnswParameters(metric=VectorSearchAlgorithmMetric.COSINE))],
            profiles=[VectorSearchProfile(name="vp", algorithm_configuration_name="hnsw")]),
    ))

    search_client = SearchClient(SEARCH_ENDPOINT, SRC_INDEX, credential)
    search_client.upload_documents([{**c, "snippet_vector": v} for c, v in zip(chunks, vectors)])
    for _ in range(30):  # The document count can take a moment to reflect the upload
        if search_client.get_document_count() == len(chunks):
            break
        time.sleep(2)

    index = index_client.get_index(SRC_INDEX)
    vector_field = next(f for f in index.fields if f.name == "snippet_vector")
    metric = index.vector_search.algorithms[0].parameters.metric
    before = {"docs": search_client.get_document_count(), "scoring_profiles": [p.name for p in (index.scoring_profiles or [])],
              "vector_flags": (vector_field.stored, vector_field.hidden)}

    check("T2-1 The document count matches the chunk count", before["docs"] == len(chunks), f"{before['docs']} / {len(chunks)}")
    check("T2-2 The vector field is stored, retrievable, and uses HNSW with cosine",
          vector_field.stored is not False and not vector_field.hidden and vector_field.vector_search_dimensions == DIMS and getattr(metric, "value", metric) == "cosine",
          f"stored={vector_field.stored} hidden={vector_field.hidden} metric={getattr(metric, 'value', metric)}")

## 4. Configuration file and T3: `preflight`

You run the executable with a **nested configuration JSON** (`--config`). It has the same shape as the configuration that the plugin writes.

| Section | Contents |
| --- | --- |
| `search` | Endpoint, API version (`2026-11-01-preview`), source index, and the vector, text, and document source fields |
| `authentication` | `defaultAzureCredential`, no interactive browser, tenant ID |
| `models` | The Azure OpenAI chat deployment used for labeling (**at most one**) |
| `execution.topicGraph` | Sampling, `maxLeafSize`, concurrency, requests per second. `readOnly: false` allows the temporary scoring profile |
| `execution.sitemap` | `enabled: true` enables stage 7 (labeling) |
| `outputDirectory` | The output location. The executable owns `topicGraph` under it |

`preflight` is a **read-only** check and is the same as `build --dry-run`. It inspects the source schema, document count, and dimensions, estimates the number of calls, and announces **the temporary changes it will make**.

It does not delete the output folder and keeps the checkpoints of an interrupted build. If an unfinished build exists, T4 resumes it with the same configuration and output destination.

> Source: [RUN-TOPICGRAPH.md (bundled with the distribution)](https://github.com/Azure/foundry-iq-tool-releases) / [native_adapter.py (topicgraph_skills wheel)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
def run_exe(*args, timeout=None):
    return subprocess.run([str(EXE), *map(str, args)], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)


if USE_EXISTING_OUTPUT:
    print("Existing output mode: the configuration is not overwritten, and neither authentication nor preflight runs.")
else:
    payload = credential.get_token("https://management.azure.com/.default").token.split(".")[1]
    TENANT_ID = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))["tid"]
    OUT.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH = OUT / "topicgraph-config.json"
    config = {
        "search": {"endpoint": SEARCH_ENDPOINT, "apiVersion": "2026-11-01-preview", "sourceIndex": SRC_INDEX,
                   "vectorField": "snippet_vector", "textField": "snippet", "docIdField": "blob_url"},
        "authentication": {"mode": "defaultAzureCredential", "allowInteractiveBrowser": False, "tenantId": TENANT_ID},
        "models": [{"kind": "azureOpenAI", "azureOpenAIParameters": {
            "resourceUri": AOAI_ENDPOINT, "deploymentId": CHAT_DEPLOYMENT, "modelName": CHAT_MODEL, "modelApiVersion": "2024-12-01-preview"}}],
        "execution": {
            "topicGraph": {"samplingPercentage": 4, "minimumSampleCount": 30000, "probeSeed": 1234, "maxLeafSize": MAX_LEAF_SIZE,
                           "concurrency": 8, "requestsPerSecond": 25, "readOnly": False},
            "sitemap": {"enabled": True, "label_title_words": 8, "label_description_words": 60, "label_representatives": 4, "label_max_completion_tokens": 4096},
        },
        "outputDirectory": str(OUT),
    }
    CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    pre = run_exe("preflight", "--config", CONFIG_PATH, timeout=600)
    print(pre.stdout[-1500:], pre.stderr)
    pre.check_returncode()
    report = json.loads((OUT / "topicGraph" / "step-00-preflight" / "report.json").read_text(encoding="utf-8"))

    check("T3-1 preflight succeeds", pre.returncode == 0, f"exit code {pre.returncode}")
    check("T3-2 The document count and dimensions match the source", report["documentCount"] == len(chunks) and report["dimensions"] == DIMS, f"{report['documentCount']} documents / {report['dimensions']} dimensions")
    check("T3-3 The vector field is judged retrievable", report["vectorFieldRetrievable"] is True)
    check("T3-4 Adding the temporary scoring profile is announced in advance", any("scoring profile" in m for m in report["plannedMutations"]), report["plannedMutations"][0][:80])
    display(pd.DataFrame([report["callEstimate"]]).T.rename(columns={0: "Estimate"}))

## 5. T4: Build and resume after an interruption

`build` runs **stages 0 to 9** in order. `--fresh` starts a new generation (only the previous successful generation is kept). If it stops midway, you can resume from the committed checkpoints with `--resume`.

This cell chooses `--resume` if an unfinished `build-summary.json` exists, and `--fresh` otherwise. Only when the structure save in stage 6 on Windows stops with a `System.IO.IOException` file sharing violation, it waits 2 seconds, then 4 seconds, and retries with `--resume` at most 2 times. For any other error, or when the retry limit is reached, it prints the full standard error and raises an exception, and it does not proceed to reading the output. If the lock persists, check which application is using the file, and then rerun T4. This notebook never terminates the process that holds the lock.

On resume, the stage status is `completed` or `reused`. `calls` is verified with the cumulative counts saved in the checkpoints. For recovery, rerun from T4 without deleting the output in T3 or recreating the source index in T2.

| Stage | Contents |
| --- | --- |
| 0 | Preflight: check the schema, document count, and dimensions |
| 1 | Probes: using each chunk as an anchor, get its nearest neighbors (kNN) with vector search. The number of anchors is `min(N, max(minimum sample count, ceil(N x sampling rate / 100)))` |
| 2 | Views: build candidate kNN graphs (`k` and a similarity threshold) and choose among them by diagnostic criteria (the share of the largest connected component and the average degree) |
| 3 | Clustering: partition with the **Leiden algorithm** (modularity). `maxLeafSize` is the target size of a leaf |
| 4 | Hierarchy: organize the partitions into a parent-child topic hierarchy |
| 5 | Leaves: for each leaf topic, get the vector statistics (centroid) and the representative chunks |
| 6 | Structure: seal and save the structure |
| 7 | Labeling: create the topic titles and descriptions with the chat model |
| 8 | Derive: derive graph attributes such as ratios |
| 9 | Export: write out the graph schema and documents (triples) |

This table is based on the help text of the distribution and the logs of past runs. The clustering implementation itself is inside the executable and its source is not public, so the details of the algorithm cannot be verified. The Leiden algorithm is a community detection method that fixes a weakness of the Louvain algorithm (it could produce poorly connected communities).

During the run, the **calls to Foundry IQ** (`Definition`, `DefinitionUpdate`, `Count`, `RandomSample`, `VectorSampling`, `BatchedLookup`, `Knn`) and the **model calls** (`Model`) are recorded.

> Source: [From Louvain to Leiden: guaranteeing well-connected communities (Traag, Waltman, van Eck)](https://arxiv.org/abs/1810.08473) / [From Local to Global: A Graph RAG Approach (Edge et al.)](https://arxiv.org/abs/2404.16130) / [RUN-TOPICGRAPH.md and `topicgraph --help` (bundled with the v1.0.10 distribution; checkpoint retention and `--resume`)](https://github.com/Azure/foundry-iq-tool-releases/releases/tag/v1.0.10)

> Failure observed in a live environment: In the Windows event log of 2026-11-19, the directory move in `StructureSealer.Seal` ended because of a file sharing violation. Running `--resume` against the same output destination confirmed that stage 6 was complete without calling the model. The process that held the lock has not been identified.

In [ ]:
if USE_EXISTING_OUTPUT:
    print("Existing output mode: build is not launched. The next cell, 5.0, loads the saved data.")
else:
    summary_path = OUT / "topicGraph" / "build-summary.json"
    build_mode = "--fresh"
    if summary_path.is_file() and json.loads(summary_path.read_text(encoding="utf-8"))["status"] != "completed":
        build_mode = "--resume"

    started = time.time()
    for attempt in range(3):
        build = run_exe("build", "--config", CONFIG_PATH, build_mode, timeout=1800)
        print("\n".join(build.stdout.splitlines()[-25:]), build.stderr)
        if build.returncode == 0:
            break
        sharing_violation = ("System.IO.IOException" in build.stderr and "StructureSealer.Seal" in build.stderr
                             and "being used by another process" in build.stderr)
        if not sharing_violation or attempt == 2:
            build.check_returncode()
        delay = 2 ** (attempt + 1)
        print(f"Detected a file lock during the structure save. Retrying with --resume in {delay} seconds.")
        time.sleep(delay)
        build_mode = "--resume"
    elapsed = time.time() - started

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    calls = summary["calls"]
    if not check("T4-1 build succeeds", summary["status"] == "completed", f"exit code {build.returncode} / {elapsed:.0f} s"):
        raise RuntimeError(f"The build did not complete. Check {summary_path} and the T4 run log.")
    check("T4-2 All stages 0 to 9 are completed or reused", [s["number"] for s in summary["stages"]] == list(range(10)) and all(s["status"] in {"completed", "reused"} for s in summary["stages"]),
          [s["number"] for s in summary["stages"] if s["status"] not in {"completed", "reused"}])
    check("T4-3 The number of kNN calls matches the chunk count (every chunk is an anchor)", calls["Knn"]["calls"] == len(chunks), f"{calls['Knn']['calls']} calls / {len(chunks)} chunks")
    check("T4-4 The model is called (labeling)", calls.get("Model", {}).get("calls", 0) >= 1, f"{calls.get('Model', {}).get('calls')} calls")
    check("T4-5 There are no failed calls", sum(c["failures"] for c in calls.values()) == 0, {k: c["failures"] for k, c in calls.items() if c["failures"]})
    (OUT / "notebook-inputs.json").write_text(json.dumps({"structureId": summary["structureId"], "chunks": chunks}, ensure_ascii=False), encoding="utf-8")
    np.save(OUT / "notebook-vectors.npy", np.asarray(vectors, dtype=np.float32), allow_pickle=False)

### 5.0 Load the saved output (the entry point for resuming)

**After the settings cell in 1.2, you can resume from this cell.** It confirms that the build is complete, and loads the latest release, the structure, the hierarchy, the labels, the chunk IDs, and the run record from disk. With only 1.2 and this cell, you can proceed to the graph in 6.5 and the wheel in 6.6. The usual checks from 5.1 onward also use this state.

A missing output or an unfinished build explicitly raises an error. The notebook does not switch to a new build automatically. Even for an old build that has no saved file for the original chunk text and embeddings, you can check the hierarchy, the membership IDs, the graph, and the wheel. When the saved files exist, the cell also verifies that the structure ID, the chunk IDs, and the vector shape match this output.

> Source: [Topic Graph native executable and output](https://github.com/Azure/foundry-iq-tool-releases) / [NumPy load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

In [ ]:
summary_path = OUT / "topicGraph" / "build-summary.json"
manifest_path = OUT / "topicGraph" / "run-manifest.json"
if not summary_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError(f"No completed build output exists: {OUT}. Check the output location. To resume an unfinished build, explicitly choose the create path with USE_EXISTING_OUTPUT=False in 1.2.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
if summary["status"] != "completed":
    raise RuntimeError("The saved build is unfinished. Check the T4 run log. Do not delete the checkpoints.")
calls = summary["calls"]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
release = next(r for r in manifest["releases"] if r["id"] == manifest["latestRelease"])
structure = next(s for s in manifest["structures"] if s["id"] == release["structureId"])
structure_dir = OUT / "topicGraph" / structure["path"]
hierarchy = json.loads((structure_dir / "hierarchy.json").read_text(encoding="utf-8"))
labels = json.loads((OUT / "topicGraph" / "step-07-labels" / f"{release['labelVariantId']}.json").read_text(encoding="utf-8"))["labels"]
chunk_keys = json.loads((structure_dir / "chunks.json").read_text(encoding="utf-8"))["keys"]
blob = (structure_dir / "leaf-members.bin").read_bytes()
if len(blob) < 12:
    raise ValueError("The header of leaf-members.bin is missing.")
magic, member_version, leaf_count = struct.unpack_from("<4sII", blob, 0)
if magic != b"TGLM" or member_version != 3:
    raise ValueError("leaf-members.bin must be in the TGLM version 3 format.")
position, leaf_members = 12, {}
for _ in range(leaf_count):
    if position + 8 > len(blob):
        raise ValueError("The membership records in leaf-members.bin are cut off.")
    leaf_id, n = struct.unpack_from("<II", blob, position)
    if leaf_id in leaf_members or position + 8 + 4 * n > len(blob):
        raise ValueError("The leaf IDs in leaf-members.bin are duplicated, or the membership data is missing.")
    leaf_members[leaf_id] = list(struct.unpack_from(f"<{n}I", blob, position + 8))
    position += 8 + 4 * n
leaf_counts = {t["id"]: t["memberCount"] for t in hierarchy["topics"] if t["isLeaf"]}
assigned_ordinals = [o for members in leaf_members.values() for o in members]
if (position != len(blob) or set(leaf_members) != set(leaf_counts)
        or any(len(members) != leaf_counts[leaf] for leaf, members in leaf_members.items())
        or sorted(assigned_ordinals) != list(range(len(chunk_keys)))
        or len(set(chunk_keys)) != len(chunk_keys)):
    raise ValueError("Chunk memberships are duplicated or missing, or they do not match the topic membership counts.")
context = json.loads((structure_dir / "context.json").read_text(encoding="utf-8"))
chunks, vectors = None, None
inputs_path, vectors_path = OUT / "notebook-inputs.json", OUT / "notebook-vectors.npy"
if inputs_path.exists() != vectors_path.exists():
    raise FileNotFoundError("Only one of the saved files for the resume text and embeddings exists. Check notebook-inputs.json and notebook-vectors.npy.")
if inputs_path.is_file():
    saved_inputs = json.loads(inputs_path.read_text(encoding="utf-8"))
    chunks = saved_inputs["chunks"]
    vectors = np.load(vectors_path, allow_pickle=False)
    if (saved_inputs["structureId"] != structure["id"] or len({c["uid"] for c in chunks}) != len(chunks)
            or {c["uid"] for c in chunks} != set(chunk_keys)
            or vectors.shape != (len(chunks), context["dimensions"]) or not np.isfinite(vectors).all()):
        raise ValueError("The saved text and embeddings for resuming do not match the saved topic structure.")
else:
    print("The original chunk text and embeddings were not saved. 6.2 shows chunk IDs, and the chunk PCA in 6.3 is skipped. They are not regenerated.")
print(f"Loaded: release {release['id']} / {structure['topics']} topics / {len(chunk_keys)} classified chunks")

### 5.1 Stage durations and the breakdown of calls

`build-summary.json` is a report that is updated on every build call. The charts show the duration of each stage, and the count and time for each type of call. It contains no source text, vectors, or credentials.

In [ ]:
stages_df = pd.DataFrame([{"No.": s["number"], "Stage": s["name"], "Status": s["status"], "Execution": s["execution"], "Seconds": s["elapsedSeconds"], "Detail": s["details"]} for s in summary["stages"]])
calls_df = pd.DataFrame(calls).T[["calls", "attempts", "retries", "failures", "seconds"]]
display(stages_df)
display(calls_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(stages_df["No."].astype(str) + " " + stages_df["Stage"], stages_df["Seconds"], color="#6baed6")
axes[0].invert_yaxis()
axes[0].set_title("Duration per stage (seconds)")
axes[1].barh(calls_df.index, calls_df["calls"], color="#fd8d3c")
axes[1].invert_yaxis()
axes[1].set_title("Number of calls per call type")
plt.tight_layout()
plt.show()

## 6. T5: Check the contents of the output

The output remains under `outputDirectory` in the following layout.

| Location | Contents |
| --- | --- |
| `topicGraph/run-manifest.json` | The list of structures and releases. The ID and state of the latest release |
| `topicGraph/step-06-structures/<id>/` | The sealed structure. `hierarchy.json` (hierarchy), `chunks.json` (chunk keys), `leaf-members.bin` (membership per leaf), `statistics.json`, `centroids.f32` (centroids) |
| `topicGraph/step-07-labels/<id>.json` | Topic titles and descriptions (labels created by the model) |
| `topicGraph/releases/<id>.*` | The graph schema, the documents (`.documents.jsonl`), and the ontology |
| `ontology/` | A copy of the sealed structure and labels, which you hand to another tool (`ontology-build`) |

Verify the latest release loaded in 5.0. For the number of input chunks, use the count in `context.json` that was saved at build time, not a Python variable from an earlier run.

T5-3 (at least 2 leaf topics) depends on `MAX_LEAF_SIZE` and the number of chunks. With a small corpus, the data might not be split. If it FAILs, make `MAX_LEAF_SIZE` or `CHUNK_CHARS` smaller and rerun.

In [ ]:
check("T5-1 The latest release is exported", release["state"] == "exported", release["id"])
check("T5-2 Every chunk belongs to a topic (coverage = 1)", release["coverage"] == 1 and structure["clustered"] == context["documentsAtPreflight"], f"coverage={release['coverage']} / classified {structure['clustered']} / input {context['documentsAtPreflight']}")
check("T5-3 At least 2 leaf topics are created", structure["leaves"] >= 2, f"topics {structure['topics']} / leaves {structure['leaves']}")
pd.DataFrame({"Value": {"Release": release["id"], "Structure": structure["id"][:16], "Topics": structure["topics"], "Leaf": structure["leaves"], "Graph documents": release["graphDocumentCount"],
                        "Labels": release["labeling"]["labels"], "Input tokens": release["labeling"]["inputTokens"], "Output tokens": release["labeling"]["outputTokens"]}})

### 6.1 Topic hierarchy and labels

`hierarchy.json` holds each topic's parent (`parent`), level (`level`, where 0 is the broadest), member count (`memberCount`), and whether it is a leaf (`isLeaf`). The labels are in a separate file. Check that every topic has a title created by the model.

In [ ]:
topics_df = pd.DataFrame(hierarchy["topics"])
topics_df["Title"] = topics_df["id"].map(lambda i: labels[str(i)]["title"])
topics_df["Label status"] = topics_df["id"].map(lambda i: labels[str(i)]["status"])
topics_df["Description"] = topics_df["id"].map(lambda i: labels[str(i)]["description"])

check("T5-4 Every topic has a title", bool((topics_df["Label status"] == "ok").all() and (topics_df["Title"].str.len() > 0).all()), topics_df["Label status"].value_counts().to_dict())
topics_df

### 6.2 Membership (which chunk went into which leaf topic)

`leaf-members.bin` is a binary file that holds the membership of each leaf topic by chunk **ordinal** (version 3 format: the identifier `TGLM`, the version, the number of leaves, and then, for each leaf, "ID, count, and the list of ordinals"). The `keys` in `chunks.json` map each ordinal to a chunk key (`uid`). This procedure reads the executable's **internal format** on the assumption of version 3, so it may change in a future version.

Check that the membership total equals the number of saved chunk IDs. If the text is also saved, the cell shows the beginning of each chunk. If it is not saved, for example for an old build, the cell says so explicitly and shows the chunk IDs.

In [ ]:
text_by_uid = {c["uid"]: c["snippet"] for c in chunks} if chunks is not None else None
check("T5-5 The internal format (TGLM version 3) can be read, and the membership total matches the chunk count",
      magic == b"TGLM" and member_version == 3 and sum(len(m) for m in leaf_members.values()) == len(chunk_keys),
      f"leaves {leaf_count} / memberships {sum(len(m) for m in leaf_members.values())} / chunks {len(chunk_keys)}")

membership = pd.DataFrame([{"Leaf": leaf, "Title": labels[str(leaf)]["title"], "Members": len(ordinals),
                            "Chunk start" if text_by_uid is not None else "Chunk ID (text not saved)":
                            " | ".join(text_by_uid[chunk_keys[o]][:28].replace("\n", " ") if text_by_uid is not None else chunk_keys[o] for o in ordinals[:3])}
                           for leaf, ordinals in leaf_members.items()])
membership

### 6.3 Plot how the topics separate in vector space (when the original embeddings are saved)

The embeddings of the original chunks are compressed to 2 dimensions with **principal component analysis** (PCA), and each leaf topic gets its own color. Compression to 2 dimensions cannot preserve the original distances exactly.

If `notebook-inputs.json` and `notebook-vectors.npy`, which a new build saves, exist, you can draw the plot from the same input even after a kernel restart. For an old build without them, this cell prints the reason and skips the plot. It does not regenerate the embeddings or fetch them again from Foundry IQ. The graph in 6.5 and the wheel in 6.6 do not depend on these embeddings.

> Source: [NumPy singular value decomposition](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)

In [ ]:
if vectors is None:
    print("Chunk PCA skipped: the original embeddings are not saved. The visualizations in 6.5 and 6.6 can still run.")
else:
    X = np.array(vectors)
    centered = X - X.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    xy = centered @ vt[:2].T

    leaf_of_uid = {chunk_keys[o]: leaf for leaf, ordinals in leaf_members.items() for o in ordinals}
    leaf_ids = [leaf_of_uid[c["uid"]] for c in chunks]
    fig, ax = plt.subplots(figsize=(9, 6))
    for leaf in sorted(leaf_members):
        mask = np.array([i == leaf for i in leaf_ids])
        ax.scatter(xy[mask, 0], xy[mask, 1], s=60, alpha=0.8, label=f"{leaf}: {labels[str(leaf)]['title'][:18]}")
    ax.set_title("Stored chunk vectors compressed to 2 dimensions with PCA, colored by leaf topic")
    ax.legend(fontsize=8, loc="best")
    plt.show()

### 6.4 Release documents (the topics written out as a graph)

`releases/<id>.documents.jsonl` is the topic hierarchy written out as **graph documents in edge-list form** (subject, predicate, object). These are the exact rows that are used when you publish to Foundry IQ (publishing is a separate step of the plugin). Look at the count for each predicate and check that the number of documents matches the manifest's record.

In [ ]:
documents = [json.loads(line) for line in (OUT / "topicGraph" / release["documentsFile"]).read_text(encoding="utf-8").splitlines() if line.strip()]
check("T5-6 The number of graph documents matches the manifest's record", len(documents) == release["graphDocumentCount"], f"{len(documents)} documents")
pd.DataFrame(documents)["predicate"].value_counts().rename("Count").to_frame()

### 6.5 Display the whole topic graph

Using the `hierarchy` and `labels` loaded in 5.0, this cell displays **all topics and their parent-child relationships**. The collection on the left is a display-only node that bundles the multiple top-level topics. Arrows point from parent to child, and the further right you go, the finer the topics. Attributes such as the title and description are checked on hover instead of being drawn as separate nodes. The vectors themselves are not drawn.

- **Color**: One color for each branch under a top-level topic. The wheel that follows uses the same colors.
- **Node size**: The more member chunks a topic has, the larger its node. It is not the number of documents.
- **Interaction**: Hover to see the ID, depth, member count, and description, and zoom and pan as you like. The layout represents the hierarchy, not embedding distances.

This cell saves **`topic-network.html`** under `OUT` and shows a link to the file. Open the link in a regular browser. Because the Plotly JavaScript and the graph data are embedded in the HTML, you need neither the Plotly-specific renderer in VS Code, a CDN, nor an internet connection. The notebook does not render the chart inline.

This cell and the next use only locally loaded data. The first time, also install `networkx` and `plotly` with the install cell in Chapter 1. Because the HTML contains the topic titles, descriptions, and so on, review the contents before you share it.

> Source: [NetworkX depth-first search](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.traversal.depth_first_search.dfs_preorder_nodes.html) / [Plotly network graphs](https://plotly.com/python/network-graphs/) / [Plotly self-contained HTML export](https://plotly.com/python/interactive-html-export/)

In [ ]:
from html import escape
from textwrap import wrap
from IPython.display import Markdown, display

import networkx as nx
import plotly.graph_objects as go
from plotly.colors import qualitative

topic_graph = nx.DiGraph()
topic_graph.add_node("collection", title="Collection", description="All classified chunks",
                     members=hierarchy["clustered"], parent="", depth=0)
for topic in hierarchy["topics"]:
    topic_id = str(topic["id"])
    parent_id = "collection" if topic["parent"] is None else str(topic["parent"])
    label = labels[topic_id]
    topic_graph.add_node(topic_id, title=label["title"], description=label["description"],
                         members=topic["memberCount"], parent=parent_id)
    topic_graph.add_edge(parent_id, topic_id)

if (not hierarchy["topics"] or len(topic_graph) != len(hierarchy["topics"]) + 1
        or not nx.is_arborescence(topic_graph)):
    raise ValueError("There are duplicate topic IDs, a missing parent, or a cycle in the hierarchy. Check hierarchy.")
if hierarchy["clustered"] <= 0:
    raise ValueError("There are no classified chunks, so the membership ratios cannot be visualized.")
for topic_id, data in topic_graph.nodes(data=True):
    children = list(topic_graph.successors(topic_id))
    if data["members"] < 0 or (children and sum(topic_graph.nodes[c]["members"] for c in children) != data["members"]):
        raise ValueError(f"The member count of topic {topic_id} does not match the sum of its child topics.")

topic_ids = list(nx.topological_sort(topic_graph))
roots = list(topic_graph.successors("collection"))
color_by_topic = {"collection": "#cbd5e1", **{r: qualitative.Plotly[i % len(qualitative.Plotly)] for i, r in enumerate(roots)}}
for topic_id in topic_ids[1:]:
    parent_id = topic_graph.nodes[topic_id]["parent"]
    topic_graph.nodes[topic_id]["depth"] = topic_graph.nodes[parent_id]["depth"] + 1
    color_by_topic.setdefault(topic_id, color_by_topic[parent_id])

topic_parents = [topic_graph.nodes[t]["parent"] for t in topic_ids]
topic_values = [topic_graph.nodes[t]["members"] for t in topic_ids]
topic_colors = [color_by_topic[t] for t in topic_ids]
topic_titles = [topic_graph.nodes[t]["title"] for t in topic_ids]
topic_hover = [
    f"<b>{escape(data['title'])}</b><br>ID: {t} / Depth: {data['depth']}<br>Members: {data['members']} chunks<br>"
    + "<br>".join(escape(line) for line in wrap(data["description"], width=38))
    for t in topic_ids for data in [topic_graph.nodes[t]]
]
leaf_order = [t for t in nx.dfs_preorder_nodes(topic_graph, "collection") if topic_graph.out_degree(t) == 0]
y_by_topic = {t: -float(i) for i, t in enumerate(leaf_order)}
for topic_id in reversed(topic_ids):
    children = list(topic_graph.successors(topic_id))
    if children:
        y_by_topic[topic_id] = float(np.mean([y_by_topic[c] for c in children]))
positions = {t: (topic_graph.nodes[t]["depth"], y_by_topic[t]) for t in topic_ids}
network_fig = go.Figure(go.Scatter(
    x=[positions[t][0] for t in topic_ids], y=[positions[t][1] for t in topic_ids],
    mode="markers+text", textposition="top center",
    text=[("" if t == "collection" else f"ID {t}<br>") + "<br>".join(escape(line) for line in wrap(title, width=24))
          for t, title in zip(topic_ids, topic_titles)],
    hovertext=topic_hover, hovertemplate="%{hovertext}<extra></extra>",
    marker=dict(size=[22 + 38 * np.sqrt(n / hierarchy["clustered"]) for n in topic_values],
                color=topic_colors, line=dict(color="white", width=2)),
))
for parent_id, topic_id in topic_graph.edges:
    network_fig.add_annotation(
        x=positions[topic_id][0], y=positions[topic_id][1],
        ax=positions[parent_id][0], ay=positions[parent_id][1],
        xref="x", yref="y", axref="x", ayref="y", text="", showarrow=True,
        arrowhead=2, arrowwidth=1.3, arrowcolor="#94a3b8", standoff=20, startstandoff=20,
    )
network_fig.update_layout(
    title=f"Full topic graph: {len(topic_ids) - 1} topics / {topic_graph.number_of_edges()} parent-child relationships",
    template="plotly_white", height=max(760, 85 * len(leaf_order)), showlegend=False, hovermode="closest",
    font=dict(family="Segoe UI, Arial, sans-serif"), margin=dict(l=80, r=120, t=90, b=50),
    xaxis=dict(visible=False, range=[min(p[0] for p in positions.values()) - 0.5, max(p[0] for p in positions.values()) + 0.5]),
    yaxis=dict(visible=False, range=[min(p[1] for p in positions.values()) - 0.7, max(p[1] for p in positions.values()) + 1.0]),
)
html_header = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Topic Graph</title>
<script>
  (() => {
    const param = new URLSearchParams(window.location.search).get("scoutTheme");
    const theme =
      param || (window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light");
    document.documentElement.setAttribute("data-theme", theme);
  })();
</script>
<style>
:root {
  color-scheme: light;
  --cp-bg: #f7f4ef;
  --cp-bg-elevated: #fcfbf8;
  --cp-surface: #ffffff;
  --cp-surface-soft: #f5f5f5;
  --cp-border: #dedede;
  --cp-border-strong: #919191;
  --cp-text: #242424;
  --cp-text-muted: #5c5c5c;
  --cp-text-soft: #6f6f6f;
  --cp-accent: #b11f4b;
  --cp-accent-hover: #9a1a41;
  --cp-accent-soft: rgba(177, 31, 75, 0.08);
  --cp-accent-fg: #ffffff;
  --cp-success: #16a34a;
  --cp-danger: #dc2626;
  --cp-warning: #f59e0b;
  --cp-link: #0078d4;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.12);
  --cp-overlay: rgba(255, 255, 255, 0.8);
  --cp-panel: rgba(255, 255, 255, 0.86);
  --cp-panel-strong: rgba(255, 255, 255, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.55);
  --cp-highlight: rgba(177, 31, 75, 0.12);
}
html[data-theme="dark"] {
  color-scheme: dark;
  --cp-bg: #3d3b3a;
  --cp-bg-elevated: #343231;
  --cp-surface: #292929;
  --cp-surface-soft: #2e2e2e;
  --cp-border: #474747;
  --cp-border-strong: #5f5f5f;
  --cp-text: #dedede;
  --cp-text-muted: #919191;
  --cp-text-soft: #b0b0b0;
  --cp-accent: #fd8ea1;
  --cp-accent-hover: #fb7b91;
  --cp-accent-soft: rgba(253, 142, 161, 0.14);
  --cp-accent-fg: #1a1a1a;
  --cp-success: #4ade80;
  --cp-danger: #f87171;
  --cp-warning: #fbbf24;
  --cp-link: #4da6ff;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.32);
  --cp-overlay: rgba(41, 41, 41, 0.88);
  --cp-panel: rgba(41, 41, 41, 0.72);
  --cp-panel-strong: rgba(41, 41, 41, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.04);
  --cp-highlight: rgba(253, 142, 161, 0.12);
}
body { margin: 0; background: var(--cp-bg); color: var(--cp-text);
       font-family: "Segoe UI", Aptos, Calibri, -apple-system, BlinkMacSystemFont, sans-serif; }
</style></head><body>
"""
html_theme = """
const style = getComputedStyle(document.documentElement);
Plotly.relayout('{plot_id}', {
    paper_bgcolor: style.getPropertyValue('--cp-bg').trim(),
    plot_bgcolor: style.getPropertyValue('--cp-bg').trim(),
    'font.color': style.getPropertyValue('--cp-text').trim(),
    'font.family': 'Segoe UI, Aptos, Calibri, sans-serif'
});
"""
network_html = OUT / "topic-network.html"
network_html.write_text(
    html_header + network_fig.to_html(include_plotlyjs=True, full_html=False,
                                      post_script=html_theme, config={"displaylogo": False}) + "</body></html>",
    encoding="utf-8",
)
display(Markdown(f"[Open the full graph in a browser]({network_html.as_uri()})"))
print("Saved to:", network_html)

### 6.6 Display the hierarchy as a topic wheel

Run it **after the cell in 6.5**. From the center outward, it arranges the collection, then the topic hierarchy, then the **individual chunks**. Using the memberships in `leaf-members.bin` and the IDs in `chunks.json` that were loaded in 5.0, it places each chunk just outside the leaf topic it belongs to. You do not need to run 6.2.

Chunks are drawn as **one segment per chunk**. Labels such as `C1` inside a segment are display numbers equal to the ordinal in `chunks.json` plus 1. Hover to see the actual chunk ID and the leaf topic it belongs to. The cell does not show the text and does no additional fetching.

The radial position of a chunk follows the hierarchy depth of its topic. Chunks of shallow leaf topics appear on the inside, and chunks of deep leaf topics appear on the outside. The cell adds no alignment to the outermost ring and no transparent segments for it.

**The angle of each sector is proportional to its member chunk count.** A parent's `memberCount` already includes the memberships of its children, so `branchvalues="total"` is specified to prevent double counting. Because each ring represents the same chunks at a different level, you must not add up the member counts of all rings to get the count for the collection. The denominator of each ratio is the number of classified chunks, not the number of source documents.

This cell saves **`topic-wheel.html`** under `OUT`. Open the displayed link in a regular browser. Like the full graph, it is a standalone HTML file with the JavaScript embedded, so it uses neither the Plotly-specific renderer nor a CDN.

On hover, in addition to the title, description, and member count, you can see the **share of the parent** and the **share of the whole collection**. Click a sector to zoom into that branch, and click the center to go back up. So that topics with duplicate titles can still be told apart, the parent-child mapping uses the unique topic ID rather than the title.

> Source: [Plotly sunburst charts and branchvalues](https://plotly.com/python/sunburst-charts/#branchvalues) / [Sunburst ids, parents, and hovertemplate](https://plotly.com/python/reference/sunburst/)

In [ ]:
wheel_nodes = {
    t: dict(parent=parent, value=value, color=color, hover=hover, kind="topic",
            label=f"Collection<br>{hierarchy['clustered']} chunks" if t == "collection" else f"{t}: {escape(title)}")
    for t, parent, value, color, hover, title in zip(topic_ids, topic_parents, topic_values, topic_colors, topic_hover, topic_titles)
}
for leaf, ordinals in leaf_members.items():
    leaf_id = str(leaf)
    for ordinal in ordinals:
        chunk_key = chunk_keys[ordinal]
        hover = (f"<b>Chunk C{ordinal + 1}</b><br>ID: {escape(chunk_key)}"
                 f"<br>Topic: {leaf_id}: {escape(topic_graph.nodes[leaf_id]['title'])}"
                 f"<br>Share of the topic: {1 / len(ordinals):.1%}"
                 f"<br>Share of the collection: {1 / hierarchy['clustered']:.1%}")
        wheel_nodes[f"chunk:{chunk_key}"] = dict(parent=leaf_id, value=1, color=color_by_topic[leaf_id],
                                                hover=hover, label=f"C{ordinal + 1}", kind="chunk")

wheel_fig = go.Figure(go.Sunburst(
    ids=list(wheel_nodes), parents=[n["parent"] for n in wheel_nodes.values()],
    values=[n["value"] for n in wheel_nodes.values()], labels=[n["label"] for n in wheel_nodes.values()],
    branchvalues="total", sort=False, maxdepth=-1,
    marker=dict(colors=[n["color"] for n in wheel_nodes.values()],
                line=dict(color="white", width=1)),
    insidetextorientation="radial", textinfo="label",
    hovertext=[n["hover"] for n in wheel_nodes.values()],
    hovertemplate=["%{hovertext}<extra></extra>" if n["kind"] == "chunk" else
                   "%{hovertext}<br>Share of parent: %{percentParent:.1%}<br>Share of collection: %{percentRoot:.1%}<extra></extra>"
                   for n in wheel_nodes.values()],
))
wheel_fig.update_layout(
    title=f"Topic wheel: collection, topic hierarchy, and {len(chunk_keys)} chunks",
    template="plotly_white", height=850, margin=dict(l=20, r=20, t=80, b=20),
    font=dict(family="Segoe UI, Arial, sans-serif"),
)
wheel_html = OUT / "topic-wheel.html"
wheel_html.write_text(
    html_header + wheel_fig.to_html(include_plotlyjs=True, full_html=False,
                                    post_script=html_theme, config={"displaylogo": False}) + "</body></html>",
    encoding="utf-8",
)
display(Markdown(f"Topic wheel HTML: [Open in a browser]({wheel_html.as_uri()})"))
print("Saved to:", wheel_html)

## 7. T6: The temporary change is restored

During the build, the executable adds a **temporary scoring profile** to the source index (presumably to take a random sample) and deletes it when it finishes. It does not change documents or fields. `profile.json` records the state of the profile that it added (the one it owns). Read the index back and check the following.

- The state of every profile is `removed`.
- The list of scoring profiles of the index is the same as before the build.
- The document count and the vector field settings (`stored` and `hidden`) are the same as before the build.

In existing output mode, only the state in the saved `profile.json` is verified, as T6-0. Because the pre-build comparison data `before` does not exist, T6-1 to T6-3, which compare against live resources, are explicitly reported as not run. No authentication or network connection is made.

If a profile remains after a failure or an interruption, you can clean up with `topicgraph cleanup` (it deletes the temporary profile and restores the vector retrieval setting).

> Source: [Plan and apply (foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md)

In [ ]:
profile = json.loads((OUT / "topicGraph" / "profile.json").read_text(encoding="utf-8"))
states = [p["state"] for p in profile["profiles"].values()]
if USE_EXISTING_OUTPUT:
    check("T6-0 In the saved record, the temporary profile is deleted", bool(states) and all(s == "removed" for s in states), states)
    print("T6-1 to T6-3 not run: they need the pre-build comparison data, so live resources are checked only for a new build.")
else:
    after = index_client.get_index(SRC_INDEX)
    after_field = next(f for f in after.fields if f.name == "snippet_vector")
    check("T6-1 The temporary scoring profile has been deleted", bool(states) and all(s == "removed" for s in states) and [p.name for p in (after.scoring_profiles or [])] == before["scoring_profiles"], states)
    check("T6-2 The document count is unchanged", search_client.get_document_count() == before["docs"], search_client.get_document_count())
    check("T6-3 The vector field settings are unchanged", (after_field.stored, after_field.hidden) == before["vector_flags"], f"stored={after_field.stored} hidden={after_field.hidden}")

## 8. T7: `build --resume` (reusing a completed build)

**This does not run in existing output mode.** Loading saved files and testing `build --resume`, which launches the executable, are separate things. The steps below are verified only on the create path.

`--resume` resumes from the committed checkpoints. For a **completed build, it validates and reuses the build** and does not call the model again. This property protects cost and time, and even when you resume a failed build, completed stages are not recomputed. Check the following.

- The exit code is 0, and in `build-summary.json` the `mode` is `resume` and the status is `completed`.
- The difference in the cumulative number of model calls before and after the resume is 0. For example, if it was 12 before the resume and is still 12 after, the number of additional calls is 0.
- The ID of the latest release does not change.

In [ ]:
if USE_EXISTING_OUTPUT:
    print("T7 not run: the existing output is only loaded, so build --resume is not launched either.")
else:
    before_resume = json.loads((OUT / "topicGraph" / "build-summary.json").read_text(encoding="utf-8"))
    model_calls_before = before_resume["calls"].get("Model", {}).get("calls", 0)
    resume = run_exe("build", "--config", CONFIG_PATH, "--resume", timeout=1800)
    print("\n".join(resume.stdout.splitlines()[-8:]), resume.stderr)
    resume.check_returncode()
    summary2 = json.loads((OUT / "topicGraph" / "build-summary.json").read_text(encoding="utf-8"))
    manifest2 = json.loads((OUT / "topicGraph" / "run-manifest.json").read_text(encoding="utf-8"))
    model_calls_after = summary2["calls"].get("Model", {}).get("calls", 0)
    additional_model_calls = model_calls_after - model_calls_before
    check("T7-1 resume succeeds and reuses the completed build", resume.returncode == 0 and summary2["status"] == "completed" and summary2["mode"] == "resume",
          f"exit code {resume.returncode} / mode={summary2['mode']}")
    check("T7-2 The model is not called again", additional_model_calls == 0, f"before the resume {model_calls_before} / after the resume {model_calls_after} / additional {additional_model_calls} calls")
    check("T7-3 The release ID does not change", manifest2["latestRelease"] == manifest["latestRelease"], manifest2["latestRelease"])
    display(pd.DataFrame([{"No.": s["number"], "Stage": s["name"], "Execution": s["execution"], "Seconds": s["elapsedSeconds"]} for s in summary2["stages"]]))

## 9. Publish to Foundry IQ, read back, and visualize

**This section is also self-contained in the same notebook.** It registers the completed local release in a dedicated index, and then draws the live data that you read back from Foundry IQ and explores it with GQL. You need neither Functional Test 1 nor 3, no additional model settings, and no Blob Storage. Connecting to a topic search profile or a knowledge base is out of scope.

To keep `Run All` limited to the local output, the default value of `RUN_FOUNDRY_IQ` is `False`. To publish and read back, set it to `True` in the next settings cell. For the first registration, review the plan, set `CONFIRM_TOPIC_WRITES = True`, and run the create cell and the cells after it.

**Reading later:** In a new kernel, you can run only **9.1 (settings and common code), then 9.4, then 9.5**. Because it uses `TOPIC_INDEX_NAME` or the saved record, the local build output is not needed. You also do not run the creation settings at the beginning.

> Source: [Topic Graph search configuration and prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

### 9.1 Target and how to run

Connect with `RUN_FOUNDRY_IQ = True`. If you specify an existing index name, the registration step is skipped. If the name is empty, the notebook checks the saved record and then the candidates named `ft-published-topics-`. If there are several candidates, you must specify one explicitly. Only when there are no candidates does it proceed to the first-registration plan. It does not overwrite or delete existing resources. The saved record keeps only the target, the API version, and the registered name.

In [ ]:
import hashlib
import json
import os
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex
from dotenv import load_dotenv

import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Segoe UI", "Arial", "DejaVu Sans"]

RUN_FOUNDRY_IQ = False
CONFIRM_TOPIC_WRITES = False
TOPIC_INDEX_NAME = ""
TOPIC_OUTPUT = Path(".functional-test") / "topicgraph-out"
PUBLISHED_STATE = Path(".functional-test") / "published-resources.json"
PREFIX = "ft-published"
API_VERSION = "2026-11-01-preview"
MAX_DISPLAY_EDGES = 200

if RUN_FOUNDRY_IQ:
    if not 1 <= MAX_DISPLAY_EDGES <= 1000:
        raise ValueError("Set MAX_DISPLAY_EDGES in the range 1 to 1000.")
    load_dotenv(override=True)
    SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)
else:
    print("Chapter 9 not run: set RUN_FOUNDRY_IQ = True to publish to or read from Foundry IQ.")

In [ ]:
if RUN_FOUNDRY_IQ:
    if not TOPIC_INDEX_NAME and PUBLISHED_STATE.is_file():
        saved_targets = json.loads(PUBLISHED_STATE.read_text(encoding="utf-8"))
        if saved_targets["endpoint"].rstrip("/") != SEARCH_ENDPOINT or saved_targets["api_version"] != API_VERSION:
            raise ValueError("The saved record differs from the current target or API version. Specify the registered name or the save location explicitly.")
        TOPIC_INDEX_NAME = saved_targets.get("topic_index", "")
    if not TOPIC_INDEX_NAME:
        topic_candidates = sorted(name for name in index_client.list_index_names() if name.startswith(f"{PREFIX}-topics-"))
        display(pd.DataFrame({"Topic Graph candidates": topic_candidates}))
        if len(topic_candidates) > 1:
            raise ValueError("Specify an existing index name in TOPIC_INDEX_NAME.")
        if topic_candidates:
            TOPIC_INDEX_NAME = topic_candidates[0]
    CREATE_TOPIC = not bool(TOPIC_INDEX_NAME)
    print("Proceeding to the first-registration plan." if CREATE_TOPIC else f"Reading the registered index: {TOPIC_INDEX_NAME}")

### Common drawing code

Node color indicates the type, and arrow labels indicate the relationship. A `MultiDiGraph` is used so that multiple relationships between the same nodes are not dropped. The Topic Graph is laid out from parent to child, from left to right; everything else uses an automatic layout for readability. The distances do not represent semantic closeness or confidence.

> Source: [NetworkX MultiDiGraph](https://networkx.org/documentation/stable/reference/classes/multidigraph.html) / [spring_layout](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.layout.spring_layout.html) / [multipartite_layout](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.layout.multipartite_layout.html)

In [ ]:
def draw_graph(graph, title, positions=None):
    if not graph:
        raise ValueError("There are no relationships to draw. Check the number of retrieved documents and the ontology.")
    kinds = sorted({data["kind"] for _, data in graph.nodes(data=True)})
    palette = {kind: plt.cm.tab20(number % 20) for number, kind in enumerate(kinds)}
    if positions is None:
        positions = nx.spring_layout(graph, seed=7, k=1.5)
    styles = [f"arc3,rad={number * 0.12}" for number in range(10)]
    fig, ax = plt.subplots(figsize=(16, 11))
    nx.draw_networkx_nodes(graph, positions, ax=ax, node_size=1800, alpha=0.9,
                           node_color=[palette[graph.nodes[node]["kind"]] for node in graph])
    nx.draw_networkx_edges(graph, positions, ax=ax, arrows=True, arrowsize=15, node_size=1800, connectionstyle=styles, edge_color="#777777")
    nx.draw_networkx_labels(graph, positions, ax=ax, font_size=9,
                            labels={node: textwrap.fill(str(data["label"]), 15) for node, data in graph.nodes(data=True)},
                            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5})
    nx.draw_networkx_edge_labels(graph, positions, ax=ax, edge_labels=nx.get_edge_attributes(graph, "predicate"),
                                 font_size=7, rotate=False, node_size=1800, connectionstyle=styles)
    ax.legend(handles=[Patch(color=palette[kind], label=kind) for kind in kinds], loc="upper left")
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### Display GQL results

Use `SearchClient.graph(gql=...)` for both the Topic Graph and the Entity Graph. The target is an index with `graphProfile.kind = edgeList`. The common display code lets you check the result table, the diagnostics, and the execution statistics, and the concrete queries run in 9.5. Running GQL directly does not call a knowledge base or Azure OpenAI.

### How to read the results and constraints

- `node.id`, `node.name`, and `edge.predicate` are the logical IDs, display names, and relationships that the index's `graphProfile.fields` defines. You do not write physical field names directly in GQL. A topic's `name` may differ from the title that is filled in from `prop:title` at drawing time.
- Narrow the search scope by type, relationship, and name. This service rejects an all-edge traversal that specifies none of them. Because names are not necessarily unique, if you want to pin down a specific entity, filter by the `id` you retrieved.
- Each example limits the number of returned rows with `LIMIT 10`. `@odata.count` is the number of returned rows, not the count for the whole graph. Execution statistics and diagnostics are also displayed, so check the warnings about estimated results and execution limits.
- Zero rows means that no path matching the condition could be retrieved. It is not proof that extraction is complete, nor that the fact does not exist in the source documents.

> Source: [Documents - Graph (request, result, diagnostics, statistics)](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [Querying the Entity Graph directly with GQL](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Topic Graph search settings](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

In [ ]:
from azure.search.documents.models import (
    GraphQueryResult,
    GraphQueryProvenanceParameters,
    GraphQueryProvenanceHydrateDocumentsParameters,
)


def display_gql(result: GraphQueryResult) -> None:
    print("Rows returned:", len(result.value))
    display(pd.json_normalize(result.as_dict()["value"]))
    if result.diagnostics:
        print("Diagnostics (warnings, errors, execution info):")
        display(pd.DataFrame([item.as_dict() for item in result.diagnostics]))
    if result.statistics:
        print("Execution statistics:")
        display(pd.DataFrame([result.statistics.as_dict()]))

In [ ]:
def require_confirmation(confirmed):
    if not confirmed:
        raise RuntimeError("Review the registration plan, the cost, and the viewing permissions, and then set this section's approval flag to True. No writes to Azure have been made.")

In [ ]:
def save_published_target(target_key, target_name):
    targets = {"endpoint": SEARCH_ENDPOINT, "api_version": API_VERSION}
    if PUBLISHED_STATE.is_file():
        targets = json.loads(PUBLISHED_STATE.read_text(encoding="utf-8"))
        if targets["endpoint"].rstrip("/") != SEARCH_ENDPOINT:
            raise ValueError("The saved record points to a different target from the current one. Use a different location for PUBLISHED_STATE.")
        if targets["api_version"] != API_VERSION:
            raise ValueError("The saved record has a different API version. Check compatibility, or use a different location for PUBLISHED_STATE.")
    targets[target_key] = target_name
    PUBLISHED_STATE.parent.mkdir(parents=True, exist_ok=True)
    PUBLISHED_STATE.write_text(json.dumps(targets, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved the registered destination for reading:", PUBLISHED_STATE.resolve())

## 9.2 Verify the release

Only the `exported` release of the completed build is targeted. `schemaFile` and `documentsFile` use the paths recorded in the release, and paths that point outside the output folder are rejected. The registered name includes the hash of the artifacts so that it is never mixed up with another release.

`@search.action` is set by the SDK, so it is removed, but if the original operation is anything other than `upload`, processing stops. The document count and duplicate keys are also checked before writing.

> Source: [Release format of the Topic Graph distribution](https://github.com/Azure/foundry-iq-tool-releases/releases/tag/v1.0.10) / [Documents - Index](https://learn.microsoft.com/rest/api/searchservice/documents/index?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_root = TOPIC_OUTPUT / "topicGraph"
    build_summary = json.loads((topic_root / "build-summary.json").read_text(encoding="utf-8"))
    if build_summary["status"] != "completed":
        raise RuntimeError("The Functional Test 2 build is unfinished. Recover it in T4 and then run this cell.")
    manifest = json.loads((topic_root / "run-manifest.json").read_text(encoding="utf-8"))
    release = next(item for item in manifest["releases"] if item["id"] == manifest["latestRelease"])
    if release["state"] != "exported":
        raise RuntimeError("The latest release is not exported.")

    schema_path = topic_root / release["schemaFile"]
    documents_path = topic_root / release["documentsFile"]
    if not all(path.resolve().is_relative_to(topic_root.resolve()) for path in (schema_path, documents_path)):
        raise ValueError("The release files point outside the Topic Graph output folder.")
    schema_payload = json.loads(schema_path.read_text(encoding="utf-8"))
    topic_documents = [json.loads(line) for line in documents_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not topic_documents or len(topic_documents) != release["graphDocumentCount"]:
        raise ValueError("The release's document count does not match the contents of documentsFile.")
    if any(document.get("@search.action") != "upload" for document in topic_documents):
        raise ValueError("Releases that contain operations other than upload are not registered.")
    topic_documents = [{key: value for key, value in document.items() if key != "@search.action"} for document in topic_documents]
    topic_hash = hashlib.sha256(schema_path.read_bytes() + documents_path.read_bytes()).hexdigest()
    TOPIC_INDEX_NAME = f"{PREFIX}-topics-{topic_hash[:12]}"
    topic_index = SearchIndex({**schema_payload, "name": TOPIC_INDEX_NAME, "description": f"graph-test-2 Topic Graph sha256:{topic_hash}"})
    if topic_index.graph_profile is None or topic_index.graph_profile.kind != "edgeList":
        raise ValueError("The release has no edgeList graph profile.")
    key_fields = [field.name for field in topic_index.fields if field.key]
    if len(key_fields) != 1 or len({document[key_fields[0]] for document in topic_documents}) != len(topic_documents):
        raise ValueError("The schema key is not unique, or document keys are duplicated.")
    display(pd.DataFrame([{"Release": release["id"], "Documents": len(topic_documents), "Destination": TOPIC_INDEX_NAME, "SHA-256": topic_hash}]))

## 9.3 Register in Foundry IQ

### Check the capacity and the registration plan

This checks only the free slot for the one topic index that this section creates. You need no resources or model settings from the other notebooks. Between the capacity check and the creation, other processes might use up the capacity.

> Source: [Foundry IQ Service Statistics](https://learn.microsoft.com/rest/api/searchservice/get-service-statistics/get-service-statistics?view=rest-searchservice-2026-11-01-preview) / [Service limits](https://learn.microsoft.com/azure/search/search-limits-quotas-capacity)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_counter = index_client.get_service_statistics().counters.index_counter
    if topic_counter is None or topic_counter.quota is None:
        raise RuntimeError("Cannot confirm the index capacity limit. Check with your administrator before writing.")
    display(pd.DataFrame([{"Target": "Index", "In use": topic_counter.usage, "Limit": topic_counter.quota, "Required now": 1}]))
    if topic_counter.quota - topic_counter.usage < 1:
        raise RuntimeError("There is not enough capacity to register the Topic Graph. Existing resources are not deleted automatically.")
    display(pd.DataFrame([
        {"Action": "Create the Topic Graph index and register documents", "Target": TOPIC_INDEX_NAME, "Details": f"{len(topic_documents)} documents"},
        {"Action": "API in use", "Target": SEARCH_ENDPOINT, "Details": API_VERSION},
    ]))

### Enable writing the Topic Graph

**Writes to Azure start with the next create and upload cells.** They use a preview API. Check the target, the registered name, the document count, the viewing permissions on the destination, and the Foundry IQ charges, and change `CONFIRM_TOPIC_WRITES = True` only if you approve. This operation does not enable writes of the Entity Graph.

If an index with the same name exists, the create API raises an error and stops. Nothing is updated or deleted. For read-only use, skip this cell and the create and upload cells, and go to 9.4.

### Create the index and register documents

This creates an index that keeps the release's `graphProfile`, vector fields, and search settings. Creating just an empty index is not completion. The following cells check the registration result for each document, wait for the changes to be reflected, and then reconcile the counts.

> Source: [Indexes - Create](https://learn.microsoft.com/rest/api/searchservice/indexes/create?view=rest-searchservice-2026-11-01-preview) / [Documents - Index](https://learn.microsoft.com/rest/api/searchservice/documents/index?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    require_confirmation(CONFIRM_TOPIC_WRITES)
    registered_topic_index = index_client.create_index(topic_index)
    print("Created:", registered_topic_index.name)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    require_confirmation(CONFIRM_TOPIC_WRITES)
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    uploaded = 0
    for start in range(0, len(topic_documents), 100):
        batch_results = topic_client.upload_documents(topic_documents[start:start + 100])
        failures = [{"Key": result.key, "Status": result.status_code, "Error": result.error_message} for result in batch_results if not result.succeeded]
        if failures:
            display(pd.DataFrame(failures))
            raise RuntimeError("Document registration failed. Resolve the errors above and then resend the same documents.")
        uploaded += len(batch_results)
    print("Number of documents accepted for registration:", uploaded)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    for attempt in range(30):
        topic_count = topic_client.get_document_count()
        if topic_count == len(topic_documents):
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"The document count has not been reflected yet. Expected {len(topic_documents)} / actual {topic_count}")
    print("PASS: the Foundry IQ document count matches the release:", topic_count)

### Save the Topic Graph destination

After the counts are confirmed to match, the target, the API version, and the topic index name are saved to `PUBLISHED_STATE`. You can save it even if the Entity Graph is not registered. Credentials and document text are not saved. Later you can read back from 9.1 and then 9.4.

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    save_published_target("topic_index", TOPIC_INDEX_NAME)

## 9.4 Read and visualize the registered data

The drawing below uses the results of `SearchClient.search()`. At most `MAX_DISPLAY_EDGES` documents are drawn, and the total count is also displayed. A topic's attributes are also counted as one document each. If you limit the display for a large release, some titles or parent-child relationships may fall outside the displayed set.

If the registered name is unspecified and there are multiple candidates, specify one of the names shown in `TOPIC_INDEX_NAME` and rerun from this read cell. The upload cell and the count reconciliation cell in 9.3 are not needed.

> Source: [Indexes - List](https://learn.microsoft.com/rest/api/searchservice/indexes/list?view=rest-searchservice-2026-11-01-preview) / [Documents - Foundry IQ](https://learn.microsoft.com/rest/api/searchservice/documents/search-post?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ:
    if not TOPIC_INDEX_NAME:
        topic_candidates = sorted(name for name in index_client.list_index_names() if name.startswith(f"{PREFIX}-topics-"))
        display(pd.DataFrame({"Topic Graph candidates": topic_candidates}))
        if len(topic_candidates) != 1:
            raise ValueError("Cannot select a single Topic Graph. Specify an existing index name in TOPIC_INDEX_NAME.")
        TOPIC_INDEX_NAME = topic_candidates[0]
    live_topic_index = index_client.get_index(TOPIC_INDEX_NAME)
    TF = live_topic_index.graph_profile.fields
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    topic_select = [field.name for field in live_topic_index.fields if field.retrievable is not False]
    topic_found = topic_client.search("*", select=topic_select, include_total_count=True, top=MAX_DISPLAY_EDGES)
    topic_total = topic_found.get_count()
    topic_frame = pd.DataFrame(list(topic_found))
    if topic_frame.empty:
        raise RuntimeError("Could not retrieve documents from the registered Topic Graph.")
    print(f"In Foundry IQ: {topic_total} documents / retrieved: {len(topic_frame)} documents")
    display(topic_frame[[TF.subject_id_field, TF.predicate_field, TF.object_id_field, TF.object_string_field]].head(12))

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_links = topic_frame[topic_frame[TF.predicate_field].isin(["HasRootTopic", "HasSubTopic"])]
    topic_titles = topic_frame[topic_frame[TF.predicate_field] == "prop:title"].set_index(TF.subject_id_field)[TF.object_string_field].to_dict()
    topic_names = topic_frame.set_index(TF.subject_id_field)[TF.subject_id_display_name_field].to_dict()
    topic_graph = nx.from_pandas_edgelist(topic_links, TF.subject_id_field, TF.object_id_field,
                                        edge_attr=TF.predicate_field, create_using=nx.MultiDiGraph())
    nx.set_edge_attributes(topic_graph, {(a, b, key): data[TF.predicate_field] for a, b, key, data in topic_graph.edges(keys=True, data=True)}, "predicate")
    for node in topic_graph:
        topic_graph.nodes[node].update(label=topic_titles.get(node, topic_names.get(node, node)), kind="Topic" if node in topic_titles else "TopicGraph")
    levels = {node: level for level, nodes in enumerate(nx.topological_generations(topic_graph)) for node in nodes}
    nx.set_node_attributes(topic_graph, levels, "level")
    positions = nx.multipartite_layout(topic_graph, subset_key="level", align="vertical")
    draw_graph(topic_graph, f"Topic Graph registered in Foundry IQ: {TOPIC_INDEX_NAME}", positions=positions)

## 9.5 Hierarchy traversal and aggregation with GQL

Run it after the GQL display code in 9.1 and the read cell in 9.4. You do not need the drawing cell or the other notebooks. You check, in order, the roots, the parent-child paths, and the number of connections per parent. This is a separate operation from the Topic Graph's diversity-aware vector search settings.

> Source: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [Topic Graph search configuration](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

### 9.5.1. Topics directly under the root

This uses the `topic_client` that the read cell in 9.4 created. `MATCH` specifies the arrow from the root to a topic, and `WHERE` narrows the result to `HasRootTopic`. `RETURN` lists the columns of the result table, and `ORDER BY` sets the order. In this example, `graph_id` is the ID of the root node.

> Source: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview). `HasRootTopic` and `HasSubTopic` are relationship names in the Topic Graph release that this notebook ingested.

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_roots_gql = """
    MATCH (root)-[edge]->(topic)
    WHERE edge.predicate = 'HasRootTopic'
    RETURN root.id AS graph_id, topic.id AS topic_id, topic.name AS topic_name
    ORDER BY topic_id
    LIMIT 10
    """
    topic_roots_result = topic_client.graph(
        gql=topic_roots_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_roots_result)

### 9.5.2. Traverse the hierarchy in 2 hops

This retrieves paths in which two arrows share the same `parent` node. The relationship to the root is `HasRootTopic`, and the relationship between a parent and a child is `HasSubTopic`. Instead of joining the results of two independent searches on the Python side, the path is specified in a single GQL query.

> Source: [Multi-hop GQL patterns](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_path_gql = """
    MATCH (root)-[first]->(parent)-[second]->(child)
    WHERE first.predicate = 'HasRootTopic' AND second.predicate = 'HasSubTopic'
    RETURN root.id AS graph_id, parent.name AS parent_topic, child.name AS child_topic
    ORDER BY parent_topic, child_topic
    LIMIT 10
    """
    topic_path_result = topic_client.graph(
        gql=topic_path_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_path_result)

### 9.5.3. Number of connections per parent topic

The `HasSubTopic` paths are aggregated by parent ID and display name. To avoid mixing different nodes that have the same display name, the ID is also included in `GROUP BY`. `COUNT(*)` is the number of matched path rows, not the total number of documents in the index. If the original graph has duplicate relationships, it is not necessarily equal to the number of deduplicated child nodes.

> Source: [Documents - Graph (running GQL and the number of returned rows)](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview). The aggregation syntax below is verified with this notebook's preview API and live data.

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_count_gql = """
    MATCH (parent)-[edge]->(child)
    WHERE edge.predicate = 'HasSubTopic'
    RETURN parent.id AS parent_id, parent.name AS parent_topic, count(*) AS child_connections
    GROUP BY parent.id, parent.name
    ORDER BY child_connections DESC, parent_id
    LIMIT 10
    """
    topic_count_result = topic_client.graph(
        gql=topic_count_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_count_result)

### Read results and cleanup

The following is the read result from 9.4. Registering in a topic index and configuring it so that it can be searched from a knowledge base are separate things. This notebook does not connect a search profile.

If you no longer need it, check the owner and the dependencies of the displayed topic index before you delete it. The destination published in this section is not deleted by the cleanup at the end. The original artifacts of Functional Test 2 and shared resources are not deleted.

> Source: [Topic Graph search configuration](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

In [ ]:
if RUN_FOUNDRY_IQ:
    display(pd.DataFrame([
        {"Artifact": "Topic Graph", "Destination": TOPIC_INDEX_NAME, "Check": f"In Foundry IQ: {topic_total} documents"},
    ]))

## 10. Results overview

For each test ID (such as `T4-1`), only the latest result is used, and the counts and details of PASS and FAIL are displayed. Even if you rerun a cell or the description of a test name changes, past results with the same ID are not counted. A test that still fails after rerunning remains as FAIL.

In [ ]:
summary_df = pd.DataFrame(results)
test_ids = summary_df["Test"].str.split(n=1).str[0]
summary_df = summary_df.loc[~test_ids.duplicated(keep="last")].reset_index(drop=True)
print(summary_df["Result"].value_counts().to_dict())
summary_df.style.apply(lambda row: ["background-color: #ffd6d6" if row["Result"] == "FAIL" else "" for _ in row], axis=1)

## 11. Cleanup (optional)

**In existing output mode, nothing is deleted even with `RUN_CLEANUP=True`.** The following describes the behavior when you explicitly enable cleanup on the create path.

It deletes the test-only index `ft-topic-src` and the working folder (the output and the downloaded executable). **If you use the output as the input of Functional Test 3, do not run it first.** Before deleting, it also checks with `topicgraph cleanup` that no temporary profile remains.

In [ ]:
if USE_EXISTING_OUTPUT:
    print("Existing output mode: the index and the working folder are not deleted.")
elif RUN_CLEANUP:
    print(run_exe("cleanup", "--config", CONFIG_PATH, timeout=600).stdout[-500:])
    index_client.delete_index(SRC_INDEX)
    shutil.rmtree(WORK, ignore_errors=True)
    print("Deleted:", SRC_INDEX, WORK)